## SPA: Puntuación por lotes

Este notebook asigna a cada cliente actual una probabilidad de irse con el modelo que tiene el alias champion en Unity Catalog. Se puntúa solo a los clientes con churn = 0, porque a los que ya se fueron no los podemos retener. Un cliente entra en la lista de riesgo cuando su probabilidad llega al umbral de 0.40 definido en config.py, que se puede cambiar sin reentrenar el modelo. Con ese umbral entra ~15% de los clientes actuales. El resultado se guarda en la tabla telco_churn.gold.customer_scores junto con la versión del modelo, el umbral y la fecha de puntuación, y se exporta como CSV al volumen telco_churn.gold.exports para usarlo en Power BI.

Limitación: la mayoría de estos clientes, ~80%, formaron parte de los datos con los que se entrenó el modelo, así que tiende a darles probabilidades de irse algo más bajas de las que tendría un cliente nuevo. En un uso real se puntuarían los clientes del mes siguiente, que el modelo no ha visto.

## ENG: Batch scoring

This notebook assigns each current customer a probability of leaving using the model that holds the champion alias in Unity Catalog. Only customers with churn = 0 are scored, because those who already left cannot be retained. A customer enters the risk list when their probability reaches the 0.40 threshold defined in config.py, which can be changed without retraining the model. With that threshold, ~15% of current customers enter the list. The result is saved to the table telco_churn.gold.customer_scores together with the model version, the threshold and the scoring date, and is exported as a CSV to the volume telco_churn.gold.exports for use in Power BI.

Limitation: most of these customers, ~80%, were part of the data used to train the model, so it tends to give them slightly lower probabilities of leaving than a new customer would get. In real use, the next month's customers, which the model has not seen, would be scored.

In [0]:
%pip install -r ../requirements.txt
dbutils.library.restartPython()

In [0]:
import mlflow
from mlflow import MlflowClient

from churn.config import GOLD_TABLE, MODEL_NAME, ID, TARGET, UMBRAL, SCORES_TABLE, SCORES_CSV, EXPORTS_VOLUME
from churn.validation import validar_gold
from datetime import datetime, timezone

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

In [0]:
pdf = spark.table(GOLD_TABLE).toPandas()
validar_gold(pdf)

In [0]:
champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
modelo = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/{champion.version}")

print(f"{MODEL_NAME}, versión {champion.version}")

In [0]:
actuales = pdf[pdf[TARGET] == 0].copy()
X = actuales.drop(columns=[ID, TARGET])

probabilidad = modelo.predict(X)[:, 1]

assert len(probabilidad) == len(actuales), f"{len(probabilidad)} predicciones para {len(actuales)} clientes"

In [0]:
print(len(probabilidad))
print(len(pdf))
print(len(pdf[pdf[TARGET] == 1]))

In [0]:
scores = actuales.drop(columns=[TARGET]).copy()
scores["churn_probability"] = probabilidad
scores["en_riesgo"] = (scores["churn_probability"] >= UMBRAL).astype(int)
scores["umbral"] = UMBRAL
scores["model_version"] = int(champion.version)
scores["scored_at"] = datetime.now(timezone.utc)

In [0]:
assert len(scores) == len(actuales), f"{len(scores)} filas para {len(actuales)} clientes"
assert scores[ID].is_unique, "customer_id duplicados"
assert scores["churn_probability"].between(0, 1).all(), "probabilidades fuera de [0, 1]"
assert not scores.isna().any().any(), "hay nulos"

In [0]:
spark.createDataFrame(scores).write.mode("overwrite").saveAsTable(SCORES_TABLE)

In [0]:
spark.table(SCORES_TABLE).printSchema()

In [0]:
scores.to_csv(SCORES_CSV, index=False)